# SAFE4ALL Climate Risk Assessment: Flood and Drought Risk Mapping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nelson1798/NOAA-Safe4All/blob/main/SAFE4ALL_Climate_Risk_Assessment_Workshop.ipynb)

## Kenya · Zimbabwe · Ghana

This guided workshop notebook produces **flood and drought risk maps and district risk statistics**. Run it from top to bottom (**Runtime -> Run all** in Colab, or **Run -> Run All Cells** in Jupyter/Anaconda).

```
 1. Select country / basin / district
          ↓
 2. Load administrative boundaries
          ↓
 3. CLIMATE          CHIRPS rainfall · ERA5-Land · TAHMO observations (optional) · TAMSAT (optional)
          ↓
 4. DROUGHT HAZARD   SPI-1 · SPI-3 · SPI-6 · rainfall anomaly · NDVI anomaly · VCI · soil-moisture anomaly
          ↓
 5. FLOOD HAZARD     extreme rainfall · Sentinel-1 flood detection · DEM · slope · HAND · flow accumulation · distance from river
          ↓
 6. EXPOSURE         WorldPop · cropland · settlements · infrastructure
          ↓
 7. VULNERABILITY    socioeconomic indicators · land use · agricultural dependence
          ↓
 8. MULTI-CRITERIA RISK MODEL
                     Flood Risk   = Flood Hazard   × Exposure × Vulnerability
                     Drought Risk = Drought Hazard × Exposure × Vulnerability
          ↓
 9. CLASSIFY         Very Low · Low · Moderate · High · Very High
          ↓
10. EXPORT           GeoTIFF · CSV · Shapefile · interactive HTML map · district risk statistics
```

| Output | Format | Where |
|---|---|---|
| Hazard, exposure, vulnerability, risk index and risk class rasters | GeoTIFF | Google Drive folder `SAFE4ALL_Flood_Drought_Risk` |
| District risk statistics | CSV | workshop cache `outputs/` + Google Drive |
| District polygons with all statistics | Shapefile (zipped) | workshop cache `outputs/` + Google Drive |
| Interactive district map and interactive raster map | HTML | workshop cache `outputs/` |
| Area and population per risk class, ranking charts, static maps | CSV / PNG | workshop cache `outputs/` |

**Important:** these are screening-level risk products, not operational forecasts or warnings. Review indicators, thresholds and weights with local experts and validate against observed events before making decisions.

## Indicator framework

| Component | Indicator | Dataset (Earth Engine ID) | Resolution |
|---|---|---|---|
| **Drought hazard** | SPI-1, SPI-3, SPI-6 – frequency of months with SPI ≤ −1 (gamma-fitted, WMO method) | CHIRPS `UCSB-CHG/CHIRPS/PENTAD` | ~5.5 km |
| | Rainfall anomaly – frequency of rainy seasons below 80 % of normal | CHIRPS | ~5.5 km |
| | NDVI anomaly – frequency of months with NDVI z-score ≤ −1 | MODIS `MODIS/061/MOD13A2` | 1 km |
| | VCI – frequency of months with VCI < 35 | MODIS | 1 km |
| | Soil-moisture anomaly – frequency of months with root-zone z-score ≤ −1 | ERA5-Land `ECMWF/ERA5_LAND/MONTHLY_AGGR` | ~11 km |
| **Flood hazard** | Extreme rainfall – mean annual maximum 1-day rainfall (Rx1day) | CHIRPS `UCSB-CHG/CHIRPS/DAILY` | ~5.5 km |
| | Sentinel-1 observed flood frequency in the flood season | `COPERNICUS/S1_GRD` | 10 m |
| | DEM (low elevation), slope, HAND, flow accumulation, distance from river | MERIT Hydro `MERIT/Hydro/v1_0_1` | 90 m |
| **Exposure** | Population | WorldPop `WorldPop/GP/100m/pop` | 100 m |
| | Cropland, settlements (cover fractions) | Copernicus Global Land Cover 2019 | 100 m |
| | Infrastructure | VIIRS night-time lights + GRIP4 roads (community catalogue, optional) | 500 m / 100 m |
| **Vulnerability** | Socioeconomic | Travel time to healthcare and cities (Oxford MAP), night-time-light deprivation, optional district poverty CSV | 1 km |
| | Land use | Hazard-specific land-cover susceptibility scores | 100 m |
| | Agricultural dependence | Cropland share within 5 km, outside built-up areas | 1 km |

Each indicator is rescaled to 0–1. Each component is the weighted sum of its indicators (weights in Section 1). Risk is the product of the three components, rescaled to 0–1 and then classified.
The code lives in the `safe4all` package (`aoi`, `drought`, `flood`, `exposure`, `vulnerability`, `model`, `export`), so each notebook cell stays short and readable.

## 0. Environment setup (run once)

This cell makes the notebook self-contained on **Colab**, local **Jupyter**, and **Anaconda** alike:

- On Colab (opened from GitHub via the badge above), only this notebook file is fetched. The cell clones the rest of the project (the `safe4all` package) into the runtime and adds it to `sys.path`.
- On a local Jupyter/Anaconda install that already has this project cloned, `safe4all` is already importable, so cloning is skipped.
- Any missing Python package is installed quietly with `pip`.

In [2]:
# --- Environment setup --------------------------------------------------------
# Safe to re-run. No separate install step or config file is required.
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Nelson1798/NOAA-Safe4All.git'
REPO_BRANCH = 'main'


def _in_colab() -> bool:
    return importlib.util.find_spec('google.colab') is not None


def _find_spec(module_name: str):
    try:
        return importlib.util.find_spec(module_name)
    except ModuleNotFoundError:
        # Raised for a dotted name whose parent package is itself missing.
        return None


if _in_colab() and _find_spec('safe4all') is None:
    # The "Open in Colab" badge only fetches this .ipynb file, so the rest of
    # the project (the safe4all package) is cloned into the runtime here.
    repo_path = Path('/content/NOAA-Safe4All')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(repo_path)],
            check=True,
        )
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

# Packages the notebook needs that are not part of a stock Colab/Jupyter
# install. Installing is a quick no-op for anything already present (for
# example after `conda env create -f environment.yml`).
REQUIRED_PACKAGES = [
    ('ee', 'earthengine-api'),
    ('geemap', 'geemap'),
    ('geopandas', 'geopandas'),
    ('folium', 'folium'),
    ('mapclassify', 'mapclassify'),
    ('matplotlib', 'matplotlib'),
    ('xarray', 'xarray'),
    ('netCDF4', 'netCDF4'),
]
missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES if _find_spec(module_name) is None]
if missing:
    print(f"Installing missing packages: {', '.join(missing)}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import safe4all  # noqa: F401  (fails loudly here if the package still can't be found)

print('Environment ready:', Path(safe4all.__file__).parent)

Installing missing packages: netCDF4
Environment ready: /content/NOAA-Safe4All/safe4all


## 1. Configure the exercise

Change only the variables in the next cell. The same study area and periods are used throughout, so validation, hazard, exposure, vulnerability and risk stay comparable.

**Area of interest (`AOI_MODE`)** – Step 1 of the workflow
- `'country'` – the whole of `COUNTRY` (default: Ghana, `ADMIN1_NAME = None`)
- `'admin1'` – one region or province (`ADMIN1_NAME`)
- `'district'` – one district (`ADMIN2_NAME`)
- `'basin'` – the HydroBASINS sub-basin containing `BASIN_POINT`

Region and district names follow FAO GAUL 2015. If a name is not found, the notebook lists the valid names.

In [ ]:
# --- Workshop choices --------------------------------------------------------
COUNTRY = 'Ghana'        # Choose: Kenya, Zimbabwe, Ghana
ADMIN1_NAME = 'Northern'       # e.g. 'Northern' (AOI_MODE = 'admin1'); None = whole country
ADMIN2_NAME = None       # e.g. 'Tamale North' (AOI_MODE = 'district')
AOI_MODE = 'admin1'     # 'country' | 'admin1' | 'district' | 'basin'
BASIN_POINT = (-0.85, 9.40)   # (lon, lat) inside the basin of interest (AOI_MODE = 'basin')
BASIN_LEVEL = 5               # HydroBASINS level; higher = smaller sub-basins

# Long-term risk analysis (climatology) and the WMO baseline used for SPI and anomalies
RISK_START_YEAR, RISK_END_YEAR = 1991, 2025
BASELINE_START_YEAR, BASELINE_END_YEAR = 1991, 2020
S1_START_YEAR, S1_END_YEAR = 2017, 2025      # Sentinel-1 flood-mapping years
SNAPSHOT_MONTH = (2025, 9)                    # month shown in the 'current conditions' maps
POPULATION_YEAR, NIGHTLIGHTS_YEAR = 2020, 2023

# Current-season window used for TAHMO validation and IMERG event context
ANALYSIS_START = '2024-03-01'
ANALYSIS_END = '2024-05-31'
EXTREME_RAIN_THRESHOLD_MM = 50
MAX_STATIONS = 10  # Start small for a workshop; increase after testing.

# Classification: 'equal' = 0.2-wide classes of the 0-1 index; 'quantile' = 20 % of the area per class
CLASS_METHOD = 'equal'
ANALYSIS_SCALE = 1000    # m - exposure/vulnerability grid, normalisation and statistics
EXPORT_SCALE = 1000      # m - GeoTIFF resolution (250-500 m for a single region or district)
EXPORT_TO_DRIVE = True
DRIVE_FOLDER = 'SAFE4ALL_Flood_Drought_Risk'

# Optional inputs
USE_TAHMO = True         # set False to skip station retrieval (no credential prompt)
USE_GRIP4_ROADS = True   # community-catalogue roads; skipped automatically if unavailable
POVERTY_CSV = None       # CSV with columns ADM2_NAME, poverty_rate (0-100) from a national survey

# Google Cloud project with the Earth Engine API enabled. Required by Earth
# Engine for every request. Pre-set to the SAFE4ALL workshop project; change
# it if you are running this against your own Cloud project instead.
EE_PROJECT = 'natural-notch-435413-j3'

# Optional: local path to a service-account JSON key for headless Earth
# Engine auth (recommended for a shared workshop project so participants
# don't each need their own OAuth login). Leave as None to use the normal
# interactive ee.Authenticate() prompt instead.
#
# IMPORTANT: never commit a key file to this repo. Keep it outside the
# project folder, or inside .safe4all-cache/ (already git-ignored), and
# point this at its local path -- or just set the EE_SERVICE_ACCOUNT_KEY /
# GOOGLE_APPLICATION_CREDENTIALS environment variable before starting
# Jupyter instead of editing this cell.
EE_SERVICE_ACCOUNT_KEY = None

# Exact Google Shared Drive name. Set this before the workshop if different.
SHARED_DRIVE_NAME = 'SAFE4ALL-workshop'

# Optional: path to an approved TAMSAT NetCDF subset for the first live run.
# It will be copied to the Shared Drive cache and reused by later runs.
TAMSAT_SOURCE_FILE = None

import copy
import os
os.environ['SAFE4ALL_SHARED_DRIVE'] = SHARED_DRIVE_NAME
os.environ['EE_PROJECT'] = EE_PROJECT
if EE_SERVICE_ACCOUNT_KEY:
    os.environ['EE_SERVICE_ACCOUNT_KEY'] = EE_SERVICE_ACCOUNT_KEY

from safe4all.config import DEFAULT_WEIGHTS, country_settings
from safe4all.aoi import aoi_label

settings = country_settings(COUNTRY)
AOI_LABEL = aoi_label(COUNTRY, AOI_MODE, ADMIN1_NAME, ADMIN2_NAME, BASIN_POINT, BASIN_LEVEL)

# Indicator weights (each group is re-scaled to sum to 1). Edit to reflect local expert judgement.
WEIGHTS = copy.deepcopy(DEFAULT_WEIGHTS)
# Example: WEIGHTS['flood_hazard']['observed_flood'] = 0.30

print('Study area:', AOI_LABEL)
print('Flood-season months:', settings['flood_months'], '| dry months:', settings['dry_months'],
      '| rainy-season months:', settings['rainy_months'])
for group, weights in WEIGHTS.items():
    print(f'{group:22s}', weights)

## 2. Data access and caching

The first successful live pull is cached. In Colab, the notebook mounts the named Shared Drive; if it is unavailable, it uses a SAFE4ALL folder in MyDrive. In local Jupyter it uses `.safe4all-cache`. All risk outputs are written to `outputs/<study area>/` inside the same cache.
TAHMO credentials are requested privately at runtime and never saved. Press Enter without typing to skip TAHMO; the risk assessment still runs.

In [1]:
import getpass
from safe4all.cache import cache_root

if USE_TAHMO and not os.environ.get('TAHMO_API_KEY'):
    os.environ['TAHMO_API_KEY'] = getpass.getpass('TAHMO API key (Enter to skip): ')
if USE_TAHMO and os.environ.get('TAHMO_API_KEY') and not os.environ.get('TAHMO_API_SECRET'):
    os.environ['TAHMO_API_SECRET'] = getpass.getpass('TAHMO API secret: ')
TAHMO_ENABLED = USE_TAHMO and bool(os.environ.get('TAHMO_API_KEY')) and bool(os.environ.get('TAHMO_API_SECRET'))

CACHE_ROOT = cache_root()
OUTPUT_DIR = CACHE_ROOT / 'outputs' / AOI_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Workshop cache: {CACHE_ROOT}')
print(f'Risk outputs:   {OUTPUT_DIR}')
print('TAHMO enabled:', TAHMO_ENABLED)

ModuleNotFoundError: No module named 'safe4all'

# STEP 1–2 · Study area and administrative boundaries

Earth Engine is initialised here. Make sure `EE_PROJECT` in Section 1 is set to a Google Cloud project with the Earth Engine API enabled. FAO GAUL 2015 provides the country (level 0), region (level 1) and district (level 2) boundaries. Districts are the reporting units for the Step 10 statistics.

In [ ]:
import geemap
from safe4all.gee import initialise_earth_engine
from safe4all.aoi import select_aoi

try:
    ee = initialise_earth_engine()
except Exception as error:
    raise RuntimeError(f'Earth Engine is not ready: {error}. The risk assessment needs Earth Engine; '
                       'complete authentication and re-run this cell.') from error

aoi = select_aoi(ee, COUNTRY, AOI_MODE, ADMIN1_NAME, ADMIN2_NAME, BASIN_POINT, BASIN_LEVEL)
study_fc, study_area, districts = aoi['study_fc'], aoi['study_area'], aoi['districts']
boundary = study_fc   # kept for the validation and TAMSAT cells
print(f"Area: {study_area.area(1000).divide(1e6).getInfo():,.0f} km2 | districts: {districts.size().getInfo()}")

from safe4all.config import BASEMAP_ATTRIBUTION, BASEMAP_NAME, BASEMAP_URL, CLASS_COLORS, CLASS_LABELS
INDEX_VIS = {'min': 0, 'max': 1, 'palette': CLASS_COLORS}
CLASS_VIS = {'min': 1, 'max': 5, 'palette': CLASS_COLORS}
LEGEND = dict(zip(CLASS_LABELS, CLASS_COLORS))
FREQ_VIS = {'min': 0.05, 'max': 0.25, 'palette': ['ffffcc', 'fd8d3c', '800026']}


def new_map():
    # Esri World Street Map basemap: OpenStreetMap tiles return '403 Access blocked'
    # and CARTO now requires an API key; Esri needs no key.
    try:
        m = geemap.Map(basemap='Esri.WorldStreetMap')
    except Exception:
        m = geemap.Map()
        m.add_tile_layer(BASEMAP_URL, name=BASEMAP_NAME, attribution=BASEMAP_ATTRIBUTION)
    m.centerObject(study_fc, aoi['zoom'])
    return m


def add_boundaries(m):
    m.addLayer(districts.style(color='555555', fillColor='00000000', width=0.6), {}, 'Districts')
    m.addLayer(study_fc.style(color='black', fillColor='00000000', width=2), {}, 'Study area')
    m.addLayerControl()


Map = new_map()
add_boundaries(Map)
display(Map)

# STEP 3 · Climate

## 3.1 CHIRPS rainfall and ERA5-Land
CHIRPS is the long-term rainfall record for SPI, rainfall anomalies and extreme rainfall. ERA5-Land provides root-zone soil moisture (0–28 cm). Both are shown as baseline climatologies.
The full Step 4 drought indicators are computed here too, because they share these inputs.

In [ ]:
from safe4all.drought import drought_indicators

drought = drought_indicators(
    ee, study_area, settings, RISK_START_YEAR, RISK_END_YEAR,
    baseline=(BASELINE_START_YEAR, BASELINE_END_YEAR), spi_scales=(1, 3, 6),
    weights=WEIGHTS['drought_hazard'], snapshot=SNAPSHOT_MONTH)

rain_vis = {'min': 400, 'max': 2200, 'palette': ['f7fbff', 'c6dbef', '6baed6', '2171b5', '08306b']}
ClimateMap = new_map()
ClimateMap.addLayer(drought['soil_moisture_mean'], {'min': 0.05, 'max': 0.40, 'palette': ['8c510a', 'f6e8c3', '01665e']},
                    'ERA5-Land mean root-zone soil moisture (m3/m3)', False)
ClimateMap.addLayer(drought['annual_rainfall'], rain_vis, f'CHIRPS mean annual rainfall {BASELINE_START_YEAR}-{BASELINE_END_YEAR} (mm)')
add_boundaries(ClimateMap)
ClimateMap.add_colorbar(rain_vis, label='Mean annual rainfall (mm)')
display(ClimateMap)

## 3.2 TAHMO station observations (optional)

TAHMO stations are the local reference. The notebook keeps the quality diagnostics and flags implausible daily totals instead of silently deleting them. If a country has few stations, that limitation is itself an important workshop finding.

In [ ]:
import pandas as pd
from safe4all.cache import cached_file
from safe4all.qc import qc_daily_precipitation
from safe4all.tahmo import save_station_metadata, station_precipitation

stations = pd.DataFrame(columns=['code', 'location.latitude', 'location.longitude'])
station_frames = []
if TAHMO_ENABLED:
    try:
        metadata_path = cached_file(
            f'tahmo/{COUNTRY}/station_metadata.csv',
            lambda path: save_station_metadata(path, COUNTRY),
        )
        stations = pd.read_csv(metadata_path).head(MAX_STATIONS)
        display(stations[['code', 'location.latitude', 'location.longitude']])
    except Exception as error:
        print(f'Could not retrieve TAHMO station metadata: {error}')
        print('Continue with satellite products, but document this coverage limitation.')

    for code in stations['code']:
        try:
            path = cached_file(
                f'tahmo/{COUNTRY}/{ANALYSIS_START}_{ANALYSIS_END}/{code}_precip.csv',
                lambda target, code=code: station_precipitation(code, ANALYSIS_START, ANALYSIS_END).to_csv(target, index=False),
            )
            frame = pd.read_csv(path)
            if not frame.empty:
                station_frames.append(frame)
        except Exception as error:
            print(f'Could not retrieve {code}: {error}')
else:
    print('TAHMO skipped (USE_TAHMO = False or no credentials).')

if station_frames:
    tahmo_daily = qc_daily_precipitation(pd.concat(station_frames, ignore_index=True))
    qc_summary = tahmo_daily.groupby('station', as_index=False).agg(
        days=('date', 'nunique'), usable_days=('usable', 'sum'),
        negative_days=('negative_flag', 'sum'), extreme_days=('extreme_flag', 'sum'),
        total_rainfall_mm=('precipitation_mm', 'sum'),
    )
    qc_summary['usable_fraction'] = qc_summary['usable_days'] / qc_summary['days']
    display(qc_summary.sort_values('usable_fraction'))
else:
    tahmo_daily = pd.DataFrame()
    qc_summary = pd.DataFrame()
    print('No station observations available. Continue with satellite products, but document this coverage limitation.')

## 3.3 Validate gridded rainfall against TAHMO

This compares period totals at station locations. Assess bias and disagreement before relying on the CHIRPS-based indicators. CHIRPS and IMERG are sampled through Earth Engine; TAMSAT is handled in the next section.

In [ ]:
from safe4all.risk import chirps_daily, imerg_event_rainfall
from safe4all.validation import rainfall_metrics

comparison = pd.DataFrame()
if not tahmo_daily.empty:
    valid_codes = qc_summary.loc[qc_summary['usable_fraction'].ge(0.8), 'station']
    point_features = [
        ee.Feature(ee.Geometry.Point([row['location.longitude'], row['location.latitude']]), {'station': row['code']})
        for _, row in stations[stations['code'].isin(valid_codes)].iterrows()
    ]
    if point_features:
        points = ee.FeatureCollection(point_features)
        region = points.geometry().buffer(50000)
        chirps_total = chirps_daily(ee, ANALYSIS_START, ANALYSIS_END, region).sum().rename('chirps_mm')
        imerg_total = imerg_event_rainfall(ee, ANALYSIS_START, ANALYSIS_END, region).rename('imerg_mm')
        # A single un-combined reducer is repeated across each band of a
        # multi-band image and each output keeps its band name, which gives
        # 'chirps_mm' and 'imerg_mm' columns.
        reducer = ee.Reducer.first()
        satellite_df = geemap.ee_to_df(chirps_total.addBands(imerg_total).reduceRegions(points, reducer, 5500))
        ground_total = tahmo_daily[tahmo_daily['usable']].groupby('station', as_index=False)['precipitation_mm'].sum()
        comparison = ground_total.merge(satellite_df[['station', 'chirps_mm', 'imerg_mm']], on='station', how='inner')
        display(comparison)
        for product in ('chirps_mm', 'imerg_mm'):
            if len(comparison) >= 3:
                print(product, rainfall_metrics(comparison['precipitation_mm'], comparison[product]))
        comparison.to_csv(OUTPUT_DIR / f'{AOI_LABEL}_TAHMO_validation.csv', index=False)
    else:
        print('No stations met the minimum QC coverage threshold.')
else:
    print('Validation runs when TAHMO observations are available.')

## 3.4 TAMSAT drought context (optional)

TAMSAT is an Africa-focused rainfall and soil-moisture product, useful for the drought narrative. For the first run, download an approved TAMSAT subset from the official data-access service and set `TAMSAT_SOURCE_FILE` in Section 1. The notebook copies it to the Shared Drive cache, so later users do not download it again. Use `rfe` for validation where available, and `rfe_filled` only for complete time-series analysis.

In [ ]:
import matplotlib.pyplot as plt
from safe4all.cache import copy_to_cache

tamsat_ds = None
tamsat_cache = CACHE_ROOT / f'tamsat/{COUNTRY}/tamsat_subset.nc'
if TAMSAT_SOURCE_FILE:
    tamsat_cache = copy_to_cache(TAMSAT_SOURCE_FILE, f'tamsat/{COUNTRY}/tamsat_subset.nc')
if tamsat_cache.exists():
    import xarray as xr
    tamsat_ds = xr.open_dataset(tamsat_cache)
    rainfall_var = 'rfe' if 'rfe' in tamsat_ds.data_vars else 'rfe_filled'
    tamsat_rainfall = tamsat_ds[rainfall_var].sel(time=slice(ANALYSIS_START, ANALYSIS_END))
    lon_min, lat_min, lon_max, lat_max = settings['bbox']
    lon_name = 'lon' if 'lon' in tamsat_rainfall.coords else 'longitude'
    lat_name = 'lat' if 'lat' in tamsat_rainfall.coords else 'latitude'
    lat_values = tamsat_rainfall[lat_name]
    lat_slice = slice(lat_min, lat_max) if float(lat_values[0]) < float(lat_values[-1]) else slice(lat_max, lat_min)
    country_tamsat = tamsat_rainfall.sel({lon_name: slice(lon_min, lon_max), lat_name: lat_slice})
    country_tamsat.sum('time').plot(figsize=(10, 6), cmap='YlGnBu', robust=True)
    plt.title(f'TAMSAT rainfall total: {COUNTRY}, {ANALYSIS_START} to {ANALYSIS_END}')
    plt.show()
else:
    print('No cached TAMSAT subset yet. Set TAMSAT_SOURCE_FILE for the first run; the rest of the notebook remains usable.')

# STEP 4 · Drought hazard

Each indicator measures **how often drought conditions occur** over 1991–2025, i.e. the fraction of months or seasons in drought. That long-term propensity is what a risk map needs. Snapshot layers show current conditions for context.

- **SPI-1, SPI-3, SPI-6** (McKee et al., 1993; WMO, 2012). For each pixel and calendar month, a gamma distribution is fitted to the baseline totals; the zero-rainfall probability *q* is included, H = q + (1 − q)·G(x), and SPI = √2·erf⁻¹(2H − 1). Dry-season months with negligible rainfall are masked, because SPI is not meaningful there. The indicator is the frequency of SPI ≤ −1.
- **Rainfall anomaly**: frequency of rainy seasons with less than 80 % of normal rainfall. The season follows the country calendar in `safe4all/config.py`, including Zimbabwe's November–March season, which crosses the year end.
- **NDVI anomaly**: frequency of months with NDVI z-score ≤ −1 (MODIS, baseline 2001–2020).
- **VCI** = 100 × (NDVI − NDVImin)/(NDVImax − NDVImin), per calendar month. The indicator is the frequency of VCI < 35, the threshold of Kogan (1995).
- **Soil-moisture anomaly**: frequency of months with an ERA5-Land root-zone z-score ≤ −1.

In [ ]:
spi_vis = {'min': -2.5, 'max': 2.5, 'palette': ['8c510a', 'd8b365', 'f6e8c3', 'f5f5f5', 'c7eae5', '5ab4ac', '01665e']}
snap_label = f'{SNAPSHOT_MONTH[0]}-{SNAPSHOT_MONTH[1]:02d}'

DroughtMap = new_map()
for k in (1, 3, 6):
    DroughtMap.addLayer(drought['snapshot'][f'spi{k}'], spi_vis, f'SPI-{k} ({snap_label})', False)
DroughtMap.addLayer(drought['snapshot']['vci'], {'min': 0, 'max': 100, 'palette': ['8c510a', 'f6e8c3', '01665e']},
                    f'VCI ({snap_label})', False)
DroughtMap.addLayer(drought['snapshot']['rainfall_pct_normal'],
                    {'min': 50, 'max': 150, 'palette': ['8c510a', 'f6e8c3', 'f5f5f5', 'c7eae5', '01665e']},
                    f'Rainy-season rainfall {RISK_END_YEAR} (% of normal)', False)
for name, image in drought['frequency'].items():
    DroughtMap.addLayer(image, FREQ_VIS, f'Drought frequency: {name}', False)
for name, image in drought['index'].items():
    DroughtMap.addLayer(image, INDEX_VIS, f'Indicator (0-1): {name}', False)
DroughtMap.addLayer(drought['hazard'], INDEX_VIS, 'Drought hazard index')
add_boundaries(DroughtMap)
DroughtMap.add_colorbar(INDEX_VIS, label='Drought hazard (0-1)')
display(DroughtMap)

# STEP 5 · Flood hazard

- **Extreme rainfall**: mean annual maximum 1-day rainfall (Rx1day), CHIRPS daily
- **Sentinel-1 flood detection**: for each year, the flood-season VV backscatter (10th percentile, speckle-filtered) is compared with a dry-season reference. A pixel is flooded when it is water-like (< −18 dB), has dropped by more than 3 dB, is not permanent water (JRC) and lies in flood-prone terrain. The indicator is the share of flood seasons in which the pixel was flooded.
- **DEM / slope / HAND / flow accumulation / distance from river**: MERIT Hydro (90 m). Low, flat terrain close to and only slightly above drainage scores highest.

The IMERG event total for `ANALYSIS_START`–`ANALYSIS_END` is added as a context layer for flood case studies.

In [ ]:
from safe4all.flood import flood_indicators

flood = flood_indicators(ee, study_area, settings, RISK_START_YEAR, RISK_END_YEAR,
                         s1_years=(S1_START_YEAR, S1_END_YEAR), weights=WEIGHTS['flood_hazard'])
imerg_event = imerg_event_rainfall(ee, ANALYSIS_START, ANALYSIS_END, study_area).clip(study_area)
topo = flood['terrain']

FloodMap = new_map()
FloodMap.addLayer(imerg_event, {'min': 0, 'max': 1000, 'palette': ['ffffcc', 'a1dab4', '41b6c4', '225ea8']},
                  f'IMERG event rainfall {ANALYSIS_START} to {ANALYSIS_END} (mm)', False)
FloodMap.addLayer(flood['rx1day'], {'min': 30, 'max': 110, 'palette': ['ffffcc', '41b6c4', '253494']}, 'Mean annual Rx1day (mm)', False)
FloodMap.addLayer(topo['dem'], {'min': 0, 'max': 800, 'palette': ['006837', 'ffffbf', 'a50026']}, 'DEM (m)', False)
FloodMap.addLayer(topo['slope'], {'min': 0, 'max': 15}, 'Slope (degrees)', False)
FloodMap.addLayer(topo['hand'], {'min': 0, 'max': 40, 'palette': ['08306b', '6baed6', 'f7fbff']}, 'HAND (m)', False)
FloodMap.addLayer(topo['upa'].log10(), {'min': 0, 'max': 5, 'palette': ['ffffff', '08519c']}, 'Flow accumulation log10(km2)', False)
FloodMap.addLayer(topo['rivers'], {'palette': ['0000ff']}, 'River network', False)
FloodMap.addLayer(flood['permanent_water'].selfMask().clip(study_area), {'palette': ['08306b']}, 'Permanent water (JRC)', False)
FloodMap.addLayer(flood['s1_latest'].selfMask(), {'palette': ['00bfff']}, f'Sentinel-1 flooded area {S1_END_YEAR}', False)
FloodMap.addLayer(flood['s1_flood_frequency'].selfMask(), {'min': 0, 'max': 0.6, 'palette': ['c6dbef', '4292c6', '08306b']},
                  f'Sentinel-1 flood frequency {S1_START_YEAR}-{S1_END_YEAR}', False)
for name, image in flood['index'].items():
    FloodMap.addLayer(image, INDEX_VIS, f'Indicator (0-1): {name}', False)
FloodMap.addLayer(flood['hazard'], INDEX_VIS, 'Flood hazard index')
add_boundaries(FloodMap)
FloodMap.add_colorbar(INDEX_VIS, label='Flood hazard (0-1)')
display(FloodMap)

# STEP 6 · Exposure

- **Population**: WorldPop counts summed to 1 km and log-transformed, so that cities do not swamp rural areas
- **Cropland** and **settlements**: Copernicus cover fractions. A cell that is 50 % or more built-up gets the maximum score.
- **Infrastructure**: VIIRS night-time lights, averaged with GRIP4 road density when that layer is available

Flood exposure weights settlements and infrastructure more; drought exposure is dominated by population and cropland.

In [ ]:
from safe4all.exposure import exposure_indicators

exposure = exposure_indicators(ee, study_area, POPULATION_YEAR, NIGHTLIGHTS_YEAR, ANALYSIS_SCALE,
                               use_roads=USE_GRIP4_ROADS, flood_weights=WEIGHTS['flood_exposure'],
                               drought_weights=WEIGHTS['drought_exposure'])

ExposureMap = new_map()
ExposureMap.addLayer(exposure['population'], {'min': 0, 'max': 2000, 'palette': ['ffffff', 'fcc5c0', '7a0177']},
                     f'Population per km2 ({POPULATION_YEAR})', False)
for name, image in exposure['index'].items():
    ExposureMap.addLayer(image, INDEX_VIS, f'Indicator (0-1): {name}', False)
ExposureMap.addLayer(exposure['drought'], INDEX_VIS, 'Drought exposure', False)
ExposureMap.addLayer(exposure['flood'], INDEX_VIS, 'Flood exposure')
add_boundaries(ExposureMap)
ExposureMap.add_colorbar(INDEX_VIS, label='Exposure (0-1)')
display(ExposureMap)

# STEP 7 · Vulnerability

- **Socioeconomic**: the mean of travel time to healthcare, remoteness (travel time to cities), night-time-light deprivation and, optionally, the district poverty rate from `POVERTY_CSV`. Each dataset gets a quick test read first. One that Earth Engine cannot serve (the Oxford healthcare layer is currently reported as corrupted) is skipped with a message, and the mean uses the rest.
- **Land use**: hazard-specific susceptibility scores per land-cover type, weighted by cover fraction. Edit the scores in `safe4all/config.py`.
- **Agricultural dependence**: share of cropland within 5 km, outside built-up areas, a proxy for rain-fed rural livelihoods

In [ ]:
from safe4all.vulnerability import vulnerability_indicators

poverty = None
if POVERTY_CSV and os.path.exists(POVERTY_CSV):
    _pov = pd.read_csv(POVERTY_CSV)
    poverty = dict(zip(_pov['ADM2_NAME'].astype(str), _pov['poverty_rate'].astype(float)))
    print(f'Poverty rates loaded for {len(poverty)} districts.')

vulnerability = vulnerability_indicators(
    ee, study_area, districts, exposure['crop_fraction'], exposure['urban_fraction'],
    exposure['nightlights_index'], ANALYSIS_SCALE, poverty_by_district=poverty,
    flood_weights=WEIGHTS['flood_vulnerability'], drought_weights=WEIGHTS['drought_vulnerability'])

VulnerabilityMap = new_map()
for name, image in vulnerability['socio'].items():
    VulnerabilityMap.addLayer(image, INDEX_VIS, f'Socioeconomic: {name}', False)
for name in ('socioeconomic', 'agricultural_dependence', 'flood_land_use', 'drought_land_use'):
    VulnerabilityMap.addLayer(vulnerability[name], INDEX_VIS, name.replace('_', ' ').capitalize(), False)
VulnerabilityMap.addLayer(vulnerability['flood'], INDEX_VIS, 'Flood vulnerability', False)
VulnerabilityMap.addLayer(vulnerability['drought'], INDEX_VIS, 'Drought vulnerability')
add_boundaries(VulnerabilityMap)
VulnerabilityMap.add_colorbar(INDEX_VIS, label='Vulnerability (0-1)')
display(VulnerabilityMap)

# STEP 8 · Multi-criteria risk model

**Flood Risk = Flood Hazard × Flood Exposure × Flood Vulnerability**
**Drought Risk = Drought Hazard × Drought Exposure × Drought Vulnerability**

The multiplicative form means risk is high only where all three components are high. For example, an uninhabited floodplain has high hazard but low risk. The product is rescaled to 0–1, which keeps the ranking, and permanent water is excluded.
The multi-hazard risk layer is the mean of flood and drought risk, for combined prioritisation.

In [ ]:
from safe4all.model import risk_index

land = flood['permanent_water'].Not()
flood_risk = risk_index(ee, flood['hazard'], exposure['flood'], vulnerability['flood'], land,
                        'flood_risk', study_area, ANALYSIS_SCALE)
drought_risk = risk_index(ee, drought['hazard'], exposure['drought'], vulnerability['drought'], land,
                          'drought_risk', study_area, ANALYSIS_SCALE)
multi_hazard_risk = flood_risk.add(drought_risk).divide(2).rename('multi_hazard_risk')

RiskMap = new_map()
for image, name in [(flood['hazard'], 'Flood hazard'), (exposure['flood'], 'Flood exposure'),
                    (vulnerability['flood'], 'Flood vulnerability'), (drought['hazard'], 'Drought hazard'),
                    (exposure['drought'], 'Drought exposure'), (vulnerability['drought'], 'Drought vulnerability'),
                    (multi_hazard_risk, 'Multi-hazard risk index'), (drought_risk, 'Drought risk index')]:
    RiskMap.addLayer(image, INDEX_VIS, name, False)
RiskMap.addLayer(flood_risk, INDEX_VIS, 'Flood risk index')
add_boundaries(RiskMap)
RiskMap.add_colorbar(INDEX_VIS, label='Risk index (0-1)')
display(RiskMap)

# STEP 9 · Classify

| Class | Label | `CLASS_METHOD = 'equal'` |
|---:|---|---|
| 1 | Very Low | 0.0 – 0.2 |
| 2 | Low | 0.2 – 0.4 |
| 3 | Moderate | 0.4 – 0.6 |
| 4 | High | 0.6 – 0.8 |
| 5 | Very High | 0.8 – 1.0 |

With `CLASS_METHOD = 'quantile'`, each class covers 20 % of the study area, for relative prioritisation. The table gives the area and population in each class.

In [ ]:
from safe4all.model import class_area_population, classify_layers

classes, breaks = classify_layers(ee, {
    'flood_hazard': flood['hazard'], 'drought_hazard': drought['hazard'],
    'flood_risk': flood_risk, 'drought_risk': drought_risk, 'multi_hazard_risk': multi_hazard_risk,
}, study_area, ANALYSIS_SCALE, CLASS_METHOD)

class_table = pd.concat([
    pd.DataFrame(class_area_population(ee, classes[name], exposure['population'], study_area, ANALYSIS_SCALE)).assign(layer=label)
    for name, label in [('flood_risk', 'Flood risk'), ('drought_risk', 'Drought risk'), ('multi_hazard_risk', 'Multi-hazard risk')]
], ignore_index=True)[['layer', 'class', 'area_km2', 'area_pct', 'population', 'population_pct']]
class_table.to_csv(OUTPUT_DIR / f'{AOI_LABEL}_risk_class_area_population.csv', index=False)
display(class_table.round({'area_km2': 0, 'area_pct': 1, 'population': 0, 'population_pct': 1}))

ClassMap = new_map()
for name in ('flood_hazard', 'drought_hazard', 'multi_hazard_risk', 'drought_risk'):
    ClassMap.addLayer(classes[name], CLASS_VIS, f"{name.replace('_', ' ').capitalize()} class", False)
ClassMap.addLayer(classes['flood_risk'], CLASS_VIS, 'Flood risk class')
add_boundaries(ClassMap)
ClassMap.add_legend(title='Risk class', legend_dict=LEGEND)
display(ClassMap)

# STEP 10 · Export

## 10.1 District risk statistics
For each district: mean hazard, exposure, vulnerability and risk; risk class; rank (1 = highest risk); and the % of area and number of people in **High or Very High** risk.

In [ ]:
from safe4all.model import district_statistics
from safe4all.export import finalise_district_table

district_fc = district_statistics(
    ee, districts,
    mean_layers={'fl_haz': flood['hazard'], 'dr_haz': drought['hazard'],
                 'fl_exp': exposure['flood'], 'dr_exp': exposure['drought'],
                 'fl_vul': vulnerability['flood'], 'dr_vul': vulnerability['drought'],
                 'fl_risk': flood_risk, 'dr_risk': drought_risk, 'mh_risk': multi_hazard_risk},
    population=exposure['population'],
    high_risk_classes={'fl': classes['flood_risk'], 'dr': classes['drought_risk']},
    scale=ANALYSIS_SCALE)

district_gdf = geemap.ee_to_gdf(district_fc.map(lambda f: f.simplify(maxError=200)))
district_gdf = district_gdf.set_crs('EPSG:4326', allow_override=True)
district_gdf, district_table = finalise_district_table(district_gdf, CLASS_METHOD)
print(f'{len(district_table)} districts')
display(district_table.head(20))

## 10.2 CSV, ranking chart and static maps

In [ ]:
from safe4all.export import save_ranking_chart, save_static_maps

csv_path = OUTPUT_DIR / f'{AOI_LABEL}_district_risk_statistics.csv'
district_table.to_csv(csv_path, index=False)
print('Saved:', csv_path)

save_ranking_chart(district_table, OUTPUT_DIR / f'{AOI_LABEL}_district_ranking.png')
save_static_maps(ee, [(classes['flood_hazard'], 'Flood hazard'), (classes['flood_risk'], 'Flood risk'),
                      (classes['drought_hazard'], 'Drought hazard'), (classes['drought_risk'], 'Drought risk')],
                 districts, study_fc, study_area, district_gdf, OUTPUT_DIR, AOI_LABEL,
                 f"Flood and drought hazard and risk - {AOI_LABEL.replace('_', ' ')}")

## 10.3 Shapefile and interactive HTML maps
- **Shapefile**: district polygons with all statistics, zipped together with a field dictionary. Field names are at most 10 characters; `fl_` / `dr_` / `mh_` stand for flood / drought / multi-hazard.
- **District HTML map**: self-contained, works offline and can be shared by e-mail.
- **Raster HTML map**: all layers at full resolution. Its Earth Engine tiles expire after a few days, so re-export it when needed.

In [ ]:
from safe4all.export import save_district_html, save_shapefile

print('Saved:', save_shapefile(district_gdf, OUTPUT_DIR, AOI_LABEL))
print('Saved:', save_district_html(district_gdf, OUTPUT_DIR / f'{AOI_LABEL}_district_risk_map.html'))

FinalMap = new_map()
for image, name in [(flood['hazard'], 'Flood hazard'), (exposure['flood'], 'Flood exposure'),
                    (vulnerability['flood'], 'Flood vulnerability'), (drought['hazard'], 'Drought hazard'),
                    (exposure['drought'], 'Drought exposure'), (vulnerability['drought'], 'Drought vulnerability'),
                    (flood_risk, 'Flood risk index'), (drought_risk, 'Drought risk index')]:
    FinalMap.addLayer(image, INDEX_VIS, name, False)
FinalMap.addLayer(classes['multi_hazard_risk'], CLASS_VIS, 'Multi-hazard risk class', False)
FinalMap.addLayer(classes['drought_risk'], CLASS_VIS, 'Drought risk class', False)
FinalMap.addLayer(classes['flood_risk'], CLASS_VIS, 'Flood risk class')
add_boundaries(FinalMap)
FinalMap.add_legend(title='Risk class', legend_dict=LEGEND)
raster_html = OUTPUT_DIR / f'{AOI_LABEL}_interactive_risk_map.html'
FinalMap.to_html(str(raster_html), title=f'Flood and drought risk - {AOI_LABEL}')
print('Saved:', raster_html)
display(FinalMap)

## 10.4 GeoTIFF and table exports to Google Drive
The exports run as Earth Engine tasks. Re-run the status cell to follow progress. Indices are exported as Float32 (0–1) and classes as Byte (1–5).

In [ ]:
from safe4all.export import export_geotiffs, export_tables

tasks = []
if EXPORT_TO_DRIVE:
    rasters = {
        'flood_hazard': flood['hazard'], 'flood_exposure': exposure['flood'], 'flood_vulnerability': vulnerability['flood'],
        'flood_risk': flood_risk, 'flood_risk_class': classes['flood_risk'],
        'drought_hazard': drought['hazard'], 'drought_exposure': exposure['drought'],
        'drought_vulnerability': vulnerability['drought'],
        'drought_risk': drought_risk, 'drought_risk_class': classes['drought_risk'],
        'multi_hazard_risk': multi_hazard_risk, 'multi_hazard_risk_class': classes['multi_hazard_risk'],
        's1_flood_frequency': flood['s1_flood_frequency'],
    }
    tasks += export_geotiffs(ee, rasters, AOI_LABEL, DRIVE_FOLDER, study_area.bounds(1000), EXPORT_SCALE)
    tasks += export_tables(ee, district_fc, AOI_LABEL, DRIVE_FOLDER)
else:
    print('EXPORT_TO_DRIVE = False - Drive exports skipped.')

In [ ]:
# Re-run this cell to check export status
for task in tasks:
    status = task.status()
    print(f"{status['description'][:60]:60s} {status['state']} {status.get('error_message', '')}")

## 10.5 Download all outputs
Packages everything in the output folder (CSV, shapefile, HTML maps, PNG figures) into one zip file.

In [ ]:
from safe4all.export import zip_outputs

zip_path = zip_outputs(OUTPUT_DIR, OUTPUT_DIR.parent / f'{AOI_LABEL}_flood_drought_risk_outputs.zip')
print('Created:', zip_path)
for file in sorted(OUTPUT_DIR.iterdir()):
    print('  -', file.name)
if _in_colab():
    from google.colab import files
    files.download(str(zip_path))

## Interpretation and next steps

1. **Hazard** shows where floods or droughts are likely. **Risk** shows where hazard coincides with exposed and vulnerable people, crops and assets. Use risk maps for prioritisation and hazard maps to explain the cause.
2. The district table ranks districts and reports the population in High/Very High risk. These are usually the most useful figures for planning.
3. Indices are **relative to the study area**: a "Very High" class in a single region is not comparable with one from a country-scale run.
4. Review TAHMO station coverage and the CHIRPS/IMERG validation (Step 3) before trusting the rainfall-based indicators.
5. Calibrate the weights, land-use scores and thresholds with local experts, e.g. through an AHP pairwise-comparison exercise.
6. Validate against observed impacts: historical flood records (EM-DAT, DesInventar, national agencies), independent flood extents (Copernicus EMS, UNOSAT), crop-yield anomalies and FEWS NET / IPC drought impacts.
7. Limitations: no hydraulic modelling of flood depth or return periods. Sentinel-1 underestimates flooding under dense vegetation and in cities. FAO GAUL 2015 district boundaries may differ from current national boundaries.

**References**: McKee et al. (1993); WMO (2012) *SPI User Guide*; Kogan (1995); Nobre et al. (2011) *HAND*; Yamazaki et al. (2019) *MERIT Hydro*; Funk et al. (2015) *CHIRPS*; Muñoz-Sabater et al. (2021) *ERA5-Land*; Weiss et al. (2018, 2020); UNDRR (2017).